#### This script uses five classifiers, linear,ann,knn,protonet and random_forest to get a final classifier using the slide aggregations.
#### It also outputs results metrics such as Balance accuracy, AUROC etc into a csv in the output folder.

In [1]:
import os
os.getcwd()

'd:\\Aamir Gulzar\\KSA_project2\\Cancer-detection-classifier\\slide_classification'

## Hugging face login

In [2]:
import os
from dotenv import load_dotenv
from huggingface_hub import login
load_dotenv()
from huggingface_hub import login

login(os.environ["HF_TOKEN"])

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: fineGrained).
Your token has been saved to C:\Users\datainsight\.cache\huggingface\token
Login successful


In [3]:
import torch
import torchvision
from torch.utils.data import DataLoader
import os
from os.path import join as j_
from PIL import Image
import pandas as pd
import numpy as np
import time
import random
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import Lambda
from PIL import Image
Image.MAX_IMAGE_PIXELS = None
# loading all packages here to start
# from dataloader import WSIDataset
from eval_patch_features.logistic import eval_linear
from eval_patch_features.ann import eval_ANN
from eval_patch_features.knn import eval_knn
from eval_patch_features.protonet import eval_protonet
from eval_patch_features.r_forest_eval import eval_r_forest
from eval_patch_features.metrics import get_eval_metrics, print_metrics
from utility import calculate_metric_averages, average_confusion_matrices, write_data_in_excel, build_probs_df
import warnings
warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


## Pipeline for TCGA Data

In [ ]:
import os
from openpyxl import load_workbook
from openpyxl.utils.dataframe import dataframe_to_rows
import torch
from torch.utils.data import Dataset
from typing import List
import pandas as pd

# --- DYNAMIC CONFIGURATIONS ---
AGGREGATION_METHODS = ["Caption_based_aggregation_15_classes"]  # ["Averaging", "Caption_based_aggregation", "Caption_based_aggregation_15_classes", "Tissue_Type_Clustering", "TITAN", "PRISM"] # Averaging , Caption_based_aggregation , Caption_based_aggregation_15_classes , Tissue_Type_Clustering , TITAN 
MODEL_NAMES = ["H-Optimus-1", "Conch1_5", "UNI2", "Virchow2", "ConchV1"] # H-Optimus-1 , Conch1_5 , UNI2 , Virchow2 , ConchV1 , PRISM                            
RESULT_FOLDER = "TCGA_Results" # "TCGA_Results_Updated"

# Define Base Dimensions for each model
MODEL_BASE_DIMS = {
    "H-Optimus-1": 1536,
    "UNI2": 1536,
    "Conch1_5": 768,
    "Virchow2": 2560,
    "ConchV1": 512, 
    "PRISM": 1280
}

# Define Multipliers for each aggregation method
AGG_MULTIPLIERS = {
    "Caption_based_aggregation": 14,
    "Caption_based_aggregation_15_classes": 15,
    "Tissue_Type_Clustering": 9,
    "Averaging": 1
}

TASK_PREFIXES = {
    "MSIH": "1-MSIH",
    "BRAF": "2-BRAF",
    "KRAS": "3-KRAS",
    "TP53": "4-TP53",
    "CIMP": "5-CIMP"
}

HIDDEN_DIM = 128
BATCH_SIZE = 4
task_names = ["MSIH"]  # "MSIH", "BRAF", "KRAS", "TP53", "CIMP"

BASE_ROOT = r"D:\Aamir Gulzar\KSA_project2"
PROJECT_ROOT = os.path.join(BASE_ROOT, "Cancer-detection-classifier", "slide_classification")

sheet_name = "baseline"

# ===========================================================================
# OUTER LOOP: Iterate over all (aggregation_method, model_name) combinations
# ===========================================================================
for AGGREGATION_METHOD in AGGREGATION_METHODS:
    for MODEL_NAME in MODEL_NAMES:

        # TITAN is only used with Conch1_5, so we skip incompatible combinations
        if AGGREGATION_METHOD == "TITAN" and MODEL_NAME != "Conch1_5":
            continue
        if AGGREGATION_METHOD == "PRISM" and MODEL_NAME != "PRISM":
            continue


        print(f"\n{'='*70}")
        print(f"  Aggregation: {AGGREGATION_METHOD} | Model: {MODEL_NAME}")
        print(f"{'='*70}\n")

        # Calculate VECTOR_DIM automatically
        # TITAN is a special case (768 regardless), so we handle it with a fallback
        if AGGREGATION_METHOD == "TITAN":
            VECTOR_DIM = 768
        elif AGGREGATION_METHOD == "PRISM":
            VECTOR_DIM = 1280
        else:
            base = MODEL_BASE_DIMS.get(MODEL_NAME, 768)
            mult = AGG_MULTIPLIERS.get(AGGREGATION_METHOD, 1)
            VECTOR_DIM = base * mult

        if AGGREGATION_METHOD == "PRISM":
            DATA_PATH = os.path.join(BASE_ROOT, "dataset", "slide_aggregation", "PRISM")
            RESULT_ROOT = os.path.join(PROJECT_ROOT, RESULT_FOLDER, "PRISM")
        else:
            DATA_PATH = os.path.join(BASE_ROOT, "dataset", "slide_aggregation", AGGREGATION_METHOD, MODEL_NAME)
            RESULT_ROOT = os.path.join(PROJECT_ROOT, RESULT_FOLDER, AGGREGATION_METHOD, MODEL_NAME)

        K_FOLDS_PATH = os.path.join(PROJECT_ROOT, "kfolds_IDARS_fixed.csv")
        # DATA_PATH = os.path.join(BASE_ROOT, "dataset", "slide_aggregation", AGGREGATION_METHOD, MODEL_NAME)
        # RESULT_ROOT = os.path.join(PROJECT_ROOT, RESULT_FOLDER, AGGREGATION_METHOD, MODEL_NAME)

        for task_name in task_names:
            folder_task_name = TASK_PREFIXES.get(task_name, task_name)

            # Build task-specific paths
            MODEL_SAVE_PATH = os.path.join(RESULT_ROOT, folder_task_name, "models")
            OUTPUT_SAVE_PATH = os.path.join(RESULT_ROOT, folder_task_name, "Output")

            # Create directories
            os.makedirs(MODEL_SAVE_PATH, exist_ok=True)
            os.makedirs(OUTPUT_SAVE_PATH, exist_ok=True)

            # Final File Paths
            EVAL_METRICS_EXCEL = os.path.join(OUTPUT_SAVE_PATH, "cross_valid_avg_eval_metrics.xlsx")
            PROBS_ALL_EXCEL = os.path.join(OUTPUT_SAVE_PATH, "cross_valid_avg_probs_all.xlsx")

            print(f"Processing Task: {task_name} using {MODEL_NAME} via {AGGREGATION_METHOD}")

            class WSIDataset(Dataset):
                def __init__(self, save_dir: str, fold_ids: List[str]):
                    self.data = []
                    self.save_dir = save_dir
                    self.fold_ids = fold_ids
                    self._load_data()

                def _load_data(self):
                    files = os.listdir(self.save_dir)
                    # Change this depending on your classification task
                    if task_name == "MSIH":
                        K_FOLDS_PATH = r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\slide_classification\kfolds_IDARS_fixed.csv"
                    elif task_name == "BRAF":
                        K_FOLDS_PATH = r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\Slide_Classification\kfold_BRAF.csv"
                    elif task_name == "KRAS":
                        K_FOLDS_PATH = r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\Slide_Classification\kfold_KRAS.csv"
                    elif task_name == "TP53":
                        K_FOLDS_PATH = r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\Slide_Classification\kfold_TP53.csv"
                    else:
                        raise ValueError(f"Unknown task name: {task_name}")

                    folds_df = pd.read_csv(K_FOLDS_PATH)

                    for wsi_file in files:
                        wsi_path = os.path.join(self.save_dir, wsi_file)

                        # Extract WSI ID without extension
                        wsi_id = os.path.splitext(wsi_file)[0]

                        # Skip files not in fold_ids
                        if wsi_id not in self.fold_ids:
                            continue

                        # Only process .pt files
                        if not wsi_path.endswith('.pt'):
                            continue

                        try:
                            # Load WSI features
                            wsi_features = torch.load(wsi_path)

                            if wsi_features.is_cuda:
                                wsi_features = wsi_features.cpu()

                            # Average or flatten features
                            if wsi_features.dim() > 1:
                                final_features = wsi_features.flatten()
                            else:
                                final_features = wsi_features

                            # Determine label based on task_name
                            if task_name == "MSIH":
                                # MSIH classification
                                if wsi_id in folds_df['WSI_Id'].values:
                                    fold_label = folds_df.loc[folds_df['WSI_Id'] == wsi_id, 'label_desc'].values[0]
                                    label = 0 if fold_label == 'nonMSIH' else 1

                            elif task_name == "BRAF":
                                # BRAF classification
                                if wsi_id in folds_df['WSI_Id'].values:
                                    fold_label = folds_df.loc[folds_df['WSI_Id'] == wsi_id, 'BRAF_mutation'].values[0]
                                    label = 0 if fold_label == 'WT' else 1

                            elif task_name == "KRAS":
                                # KRAS classification
                                if wsi_id in folds_df['WSI_Id'].values:
                                    fold_label = folds_df.loc[folds_df['WSI_Id'] == wsi_id, 'KRAS_mutation'].values[0]
                                    label = 0 if fold_label == 'WT' else 1

                            elif task_name == "TP53":
                                # TP53 classification
                                if wsi_id in folds_df['WSI_Id'].values:
                                    fold_label = folds_df.loc[folds_df['WSI_Id'] == wsi_id, 'TP53_mutation'].values[0]
                                    label = 0 if fold_label == 'WT' else 1

                            else:
                                raise ValueError(f"Unknown task name: {task_name}")

                            # Store in dataset
                            self.data.append((final_features, label, wsi_id))

                        except Exception as e:
                            print(f"[ERROR] Loading failed for {wsi_path}: {e}")

                def __len__(self):
                    return len(self.data)

                def __getitem__(self, idx):
                    features, label, wsi_id = self.data[idx]
                    return features, label, wsi_id


            # Pipeline 2
            from itertools import product

            def train_and_evaluate(fold, train_loader, val_loader, test_loader, model_type='linear'):
                def get_feats_labels(loader):
                    feats, labels, ids = [], [], []
                    for features, lbls, wsi_ids in loader:
                        feats.append(features)
                        labels.append(lbls)
                        if isinstance(wsi_ids, (list, tuple)):
                            ids.extend(wsi_ids)
                        else:
                            ids.append(wsi_ids)
                    return torch.cat(feats), torch.cat(labels), ids

                train_feats, train_labels, _ = get_feats_labels(train_loader)
                val_feats, val_labels, _ = get_feats_labels(val_loader)
                test_feats, test_labels, all_test_ids = get_feats_labels(test_loader)

                best_metrics = None
                best_dump = None
                best_params = None

                if model_type == 'lin':
                    param_grid = {
                        'C': [10],
                        'max_iter': [300]
                    }
                    for C, max_iter in product(param_grid['C'], param_grid['max_iter']):
                        eval_metrics, eval_dump = eval_linear(
                            fold=fold,
                            train_feats=train_feats,
                            train_labels=train_labels,
                            valid_feats=val_feats,
                            valid_labels=val_labels,
                            test_feats=test_feats,
                            test_labels=test_labels,
                            max_iter=max_iter,
                            save_path=MODEL_SAVE_PATH,
                            C=C,
                            verbose=False,
                        )
                        if not best_metrics or eval_metrics['lin_macro_f1'] > best_metrics['lin_macro_f1']:
                            best_metrics = eval_metrics
                            best_dump = eval_dump
                            best_params = (C, max_iter)

                elif model_type == 'ann':
                    param_grid = {
                        'hidden_dim1': [128, 256],
                        'hidden_dim2': [64, 128],
                        'max_iter': [500]
                    }
                    for h1, h2, max_iter in product(param_grid['hidden_dim1'], param_grid['hidden_dim2'], param_grid['max_iter']):
                        eval_metrics, eval_dump = eval_ANN(
                            fold=fold,
                            train_feats=train_feats,
                            train_labels=train_labels,
                            valid_feats=val_feats,
                            valid_labels=val_labels,
                            test_feats=test_feats,
                            test_labels=test_labels,
                            input_dim=VECTOR_DIM,
                            hidden_dim=h1,
                            hidden_dim2=h2,
                            max_iter=max_iter,
                            model_save_path=MODEL_SAVE_PATH,
                            verbose=False,
                        )
                        if not best_metrics or eval_metrics['ann_macro_f1'] > best_metrics['ann_macro_f1']:
                            best_metrics = eval_metrics
                            best_dump = eval_dump
                            best_params = (h1, h2, max_iter)

                elif model_type == 'knn':
                    for k in [3]:
                        eval_metrics, eval_dump = eval_knn(
                            fold=fold,
                            train_feats=train_feats,
                            train_labels=train_labels,
                            val_feats=val_feats,
                            val_labels=val_labels,
                            test_feats=test_feats,
                            test_labels=test_labels,
                            n_neighbors=k,
                            normalize_feats=True,
                            model_save_path=MODEL_SAVE_PATH,
                            verbose=False
                        )
                        if not best_metrics or eval_metrics['knn_macro_f1'] > best_metrics['knn_macro_f1']:
                            best_metrics = eval_metrics
                            best_dump = eval_dump
                            best_params = k

                elif model_type == 'proto':
                    eval_metrics, eval_dump = eval_protonet(
                        fold=fold,
                        train_feats=train_feats,
                        train_labels=train_labels,
                        val_feats=val_feats,
                        val_labels=val_labels,
                        test_feats=test_feats,
                        test_labels=test_labels,
                        normalize_feats=True,
                        model_save_path=MODEL_SAVE_PATH,
                    )
                    best_metrics = eval_metrics
                    best_dump = eval_dump
                    best_params = None

                elif model_type == 'rf':
                    param_grid = {
                        'n_estimators': [500],
                        'max_depth': [None],
                        'min_samples_split': [5],
                        'min_samples_leaf': [1],
                        'class_weight': [{0: 1, 1: 10}],
                        'prediction_threshold': [0.3]
                    }
                    for n_est, max_d, min_split, min_leaf, cw, thresh in product(
                        param_grid['n_estimators'],
                        param_grid['max_depth'],
                        param_grid['min_samples_split'],
                        param_grid['min_samples_leaf'],
                        param_grid['class_weight'],
                        param_grid['prediction_threshold']
                    ):
                        eval_metrics, eval_dump = eval_r_forest(
                            fold=fold,
                            train_feats=train_feats,
                            train_labels=train_labels,
                            valid_feats=val_feats,
                            valid_labels=val_labels,
                            test_feats=test_feats,
                            test_labels=test_labels,
                            n_estimators=n_est,
                            max_depth=max_d,
                            min_samples_split=min_split,
                            min_samples_leaf=min_leaf,
                            class_weight=cw,
                            prediction_threshold=thresh,
                            save_path=MODEL_SAVE_PATH,
                            verbose=False,
                        )
                        if not best_metrics or eval_metrics['rf_macro_f1'] > best_metrics['rf_macro_f1']:
                            best_metrics = eval_metrics
                            best_dump = eval_dump
                            best_params = (n_est, max_d, min_split, min_leaf, cw, thresh)

                else:
                    raise ValueError(f"Unsupported model type: {model_type}")

                if best_params:
                    print(f"Fold {fold} | Best Params for {model_type}: {best_params}")
                return best_metrics, best_dump, all_test_ids

            from typing import List
            def run_k_fold_cross_validation(save_dir: str, folds: List[List[str]], model_type: str = 'lin'):
                results_per_fold = []
                num_folds = len(folds)

                for i in range(num_folds):
                    # Define test and validation folds
                    test_ids = folds[i]
                    val_ids = folds[(i + 1) % num_folds]

                    # Use remaining folds as training
                    train_ids = []
                    for j in range(num_folds):
                        if j != i and j != (i + 1) % num_folds:
                            train_ids.extend(folds[j])
                    print(f"Running Fold {i + 1} with model {model_type}...")
                    print(f"  Train IDs count: {len(train_ids)}, Val IDs count: {len(val_ids)}, Test IDs count: {len(test_ids)}")
                    # Create datasets and loaders
                    train_dataset = WSIDataset(save_dir, train_ids)
                    val_dataset = WSIDataset(save_dir, val_ids)
                    test_dataset = WSIDataset(save_dir, test_ids)
                    print(f"  Dataset sizes - Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")
                    if len(train_dataset) == 0 or len(val_dataset) == 0 or len(test_dataset) == 0:
                        print(f"  ERROR: Empty dataset detected!")
                        print(f"  Sample train IDs: {train_ids[:3] if train_ids else 'None'}")
                        print(f"  Files in directory: {os.listdir(save_dir)[:5]}")
                        raise ValueError("Empty dataset - check WSI ID matching with files in directory")
                    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
                    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
                    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
                    eval_metrics, eval_dump, all_test_ids = train_and_evaluate(i, train_loader, val_loader, test_loader, model_type=model_type)
                    print_metrics(eval_metrics)
                    result = {
                        **eval_metrics,
                        **eval_dump,
                        "wsi_ids": all_test_ids,
                        "fold": i + 1
                    }
                    results_per_fold.append(result)
                return results_per_fold

            # Load CSV containing fold and wsi_id
            folds_df = pd.read_csv(K_FOLDS_PATH)

            # Group wsi_ids by fold
            num_folds = folds_df['fold'].nunique()
            folds = []
            for fold_num in sorted(folds_df['fold'].unique()):
                wsi_ids = folds_df[folds_df['fold'] == fold_num]['WSI_Id'].tolist()
                folds.append(wsi_ids)

            # Run k-fold cross-validation with different models
            model_types = ['lin', 'ann', 'knn', 'proto', 'rf']
            metric_indices = {
                'acc': 0,
                'bacc': 1,
                'macro_f1': 2,
                'weighted_f1': 3,
                'auroc': 4
            }

            eval_metrics__for_excel = []
            probs_all_for_excel = None
            for model in model_types:
                predictions_list = []
                print(f"\n\n ********* Training with model: {model}********* \n\n")
                k_folds_results = run_k_fold_cross_validation(DATA_PATH, folds, model_type=model)
                model_df = build_probs_df(k_folds_results, model_name=model)
                # === Merge predictions across models ===
                if probs_all_for_excel is None:
                    probs_all_for_excel = model_df
                else:
                    probs_all_for_excel = pd.merge(probs_all_for_excel, model_df, on=["Fold", "WSI_ID", "Target"], how="outer")

                # === Average metrics (only pass metric parts of result dicts)
                average_results = calculate_metric_averages(
                    [{k: v for k, v in result.items() if k in [f"{model}_{m}" for m in metric_indices.keys()]}
                    for result in k_folds_results],
                    metric_indices,
                    model_prefix=model
                )
                # === Confusion matrices
                confusion_matrices = [np.array(result[f"{model}_conf_matrix"]) for result in k_folds_results if f"{model}_conf_matrix" in result]

                avg_conf_matrix = average_confusion_matrices(confusion_matrices)
                print("\n\n Average results for all folds:")
                for metric, value in average_results.items():
                    print(f"{metric}: {value:.4f}")
                # Append per metric rows for each fold + average
                for metric in metric_indices.keys():
                    row = [f"{model}_{metric}"]
                    for result in k_folds_results:
                        row.append(result.get(f"{model}_{metric}", 'N/A'))
                    row.append(average_results.get(f"{model}_{metric}", 'N/A'))
                    eval_metrics__for_excel.append(row)

                # Append confusion matrix as string (per fold)
                row = [f"{model}_conf_matrix"]
                for result in k_folds_results:
                    row.append(str(result.get(f"{model}_conf_matrix", "N/A")))
                row.append(str(avg_conf_matrix))
                eval_metrics__for_excel.append(row)

            eval_metrics_df = pd.DataFrame(eval_metrics__for_excel,
                                    columns=["Metric", "Fold1", "Fold2", "Fold3", "Fold4", "AvgFolds"])
            write_data_in_excel(EVAL_METRICS_EXCEL, eval_metrics_df, sheet_name=sheet_name)
            write_data_in_excel(PROBS_ALL_EXCEL, probs_all_for_excel, sheet_name=sheet_name)


  Aggregation: Caption_based_aggregation_15_classes | Model: H-Optimus-1

Processing Task: MSIH using H-Optimus-1 via Caption_based_aggregation_15_classes


 ********* Training with model: lin********* 


Running Fold 1 with model lin...
  Train IDs count: 215, Val IDs count: 98, Test IDs count: 103
  Dataset sizes - Train: 213, Val: 97, Test: 103
Fold 0 | Best Params for lin: (10, 300)
lin_acc: 0.9126
lin_bacc: 0.8106
lin_macro_f1: 0.8194
lin_weighted_f1: 0.9114
lin_auroc: 0.9598
lin_conf_matrix: [[84  4]
 [ 5 10]]
Running Fold 2 with model lin...
  Train IDs count: 206, Val IDs count: 112, Test IDs count: 98
  Dataset sizes - Train: 204, Val: 112, Test: 97
Fold 1 | Best Params for lin: (10, 300)
lin_acc: 0.9381
lin_bacc: 0.8017
lin_macro_f1: 0.8462
lin_weighted_f1: 0.9332
lin_auroc: 0.9322
lin_conf_matrix: [[83  1]
 [ 5  8]]
Running Fold 3 with model lin...
  Train IDs count: 201, Val IDs count: 103, Test IDs count: 112
  Dataset sizes - Train: 200, Val: 101, Test: 112
Fold 2 | Best

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'D:\\Aamir Gulzar\\KSA_project2\\dataset\\slide_aggregation\\Caption_based_aggregation_15_classes\\PRISM'

<h1>Pipeline for PAIP Dataset<h1>

In [5]:
import os
from openpyxl import load_workbook
from openpyxl.utils.dataframe import dataframe_to_rows
import torch
from itertools import product
from torch.utils.data import Dataset, DataLoader
from typing import List
import pandas as pd

# --- DYNAMIC CONFIGURATIONS ---
AGGREGATION_METHODS = ["Caption_based_aggregation_15_classes"] # ["Averaging", "Caption_based_aggregation", "Caption_based_aggregation_15_classes", "Tissue_Type_Clustering", "TITAN", "PRISM"]
MODEL_NAMES = ["H-Optimus-1", "Conch1_5", "UNI2", "Virchow2","ConchV1"]  # "H-Optimus-1", "Conch1_5", "UNI2", "Virchow2","ConchV1", "PRISM"
RESULT_FOLDER = "PAIP_Results" # "PAIP_Results_Updated"

# Define Base Dimensions for each model
MODEL_BASE_DIMS = {
    "H-Optimus-1": 1536,
    "UNI2": 1536,
    "Conch1_5": 768,
    "Virchow2": 2560,
    "ConchV1": 512, 
    "PRISM": 1280
}

# Define Multipliers for each aggregation method
AGG_MULTIPLIERS = {
    "Caption_based_aggregation": 14,
    "Caption_based_aggregation_15_classes": 15,
    "Tissue_Type_Clustering": 9,
    "Averaging": 1
}

TASK_PREFIXES = {
    "MSIH": "1-MSIH",
    "BRAF": "2-BRAF",
    "KRAS": "3-KRAS",
    "TP53": "4-TP53",
    "CIMP": "5-CIMP"
}

HIDDEN_DIM = 128
BATCH_SIZE = 4
task_names = ["MSIH"]  # "MSIH", "BRAF", "KRAS", "TP53", "CIMP"

BASE_ROOT = r"D:\Aamir Gulzar\KSA_project2"
PROJECT_ROOT = os.path.join(BASE_ROOT, "Cancer-detection-classifier", "slide_classification")

sheet_name = "baseline"

# ===========================================================================
# OUTER LOOP: Iterate over all (aggregation_method, model_name) combinations
# ===========================================================================
for AGGREGATION_METHOD in AGGREGATION_METHODS:
    for MODEL_NAME in MODEL_NAMES:

        # TITAN is only used with Conch1_5, so we skip incompatible combinations
        if AGGREGATION_METHOD == "TITAN" and MODEL_NAME != "Conch1_5":
            continue
        if AGGREGATION_METHOD == "PRISM" and MODEL_NAME != "PRISM":
            continue

        print(f"\n{'='*70}")
        print(f"  Aggregation: {AGGREGATION_METHOD} | Model: {MODEL_NAME}")
        print(f"{'='*70}\n")

        # Calculate VECTOR_DIM automatically
        # TITAN is a special case (768 regardless), so we handle it with a fallback
        if AGGREGATION_METHOD == "TITAN":
            VECTOR_DIM = 768
        elif AGGREGATION_METHOD == "PRISM":
            VECTOR_DIM = 1280
        else:
            base = MODEL_BASE_DIMS.get(MODEL_NAME, 768)
            mult = AGG_MULTIPLIERS.get(AGGREGATION_METHOD, 1)
            VECTOR_DIM = base * mult

        K_FOLDS_PATH = os.path.join(PROJECT_ROOT, "paip_kfolds_71.csv")
        if AGGREGATION_METHOD == "PRISM":
            DATA_PATH = os.path.join(BASE_ROOT, "paip_data", "slide_aggregation", "PRISM")
            RESULT_ROOT = os.path.join(PROJECT_ROOT, RESULT_FOLDER, "PRISM")
        else:
            DATA_PATH = os.path.join(BASE_ROOT, "paip_data", "slide_aggregation", AGGREGATION_METHOD, MODEL_NAME)
            RESULT_ROOT = os.path.join(PROJECT_ROOT, RESULT_FOLDER, AGGREGATION_METHOD, MODEL_NAME)

        for task_name in task_names:
            folder_task_name = TASK_PREFIXES.get(task_name, task_name)

            # Build task-specific paths
            MODEL_SAVE_PATH = os.path.join(RESULT_ROOT, folder_task_name, "models")
            OUTPUT_SAVE_PATH = os.path.join(RESULT_ROOT, folder_task_name, "Output")

            os.makedirs(MODEL_SAVE_PATH, exist_ok=True)
            os.makedirs(OUTPUT_SAVE_PATH, exist_ok=True)

            EVAL_METRICS_EXCEL = os.path.join(OUTPUT_SAVE_PATH, "cross_valid_avg_eval_metrics.xlsx")
            PROBS_ALL_EXCEL = os.path.join(OUTPUT_SAVE_PATH, "cross_valid_avg_probs_all.xlsx")

            # ---------------------------------------------------------------
            # YOUR ORIGINAL CODE BELOW — COMPLETELY UNCHANGED
            # ---------------------------------------------------------------

            # ============================================
            # DATASET CLASS
            # ============================================
            class WSIDataset(Dataset):
                def __init__(self, save_dir: str, fold_ids: List[str]):
                    self.data = []
                    self.save_dir = save_dir
                    self.fold_ids = fold_ids
                    self._load_data()

                def _load_data(self):
                    files = os.listdir(self.save_dir)

                    # Load fold CSV based on task
                    if task_name == "MSIH":
                        K_FOLDS_PATH = r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\slide_classification\paip_kfolds_71.csv"
                    elif task_name == "BRAF":
                        K_FOLDS_PATH = r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\Slide_Classification\kfold_BRAF.csv"
                    elif task_name == "KRAS":
                        K_FOLDS_PATH = r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\Slide_Classification\kfold_KRAS.csv"
                    elif task_name == "TP53":
                        K_FOLDS_PATH = r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\Slide_Classification\kfold_TP53.csv"
                    else:
                        raise ValueError(f"Unknown task name: {task_name}")

                    folds_df = pd.read_csv(K_FOLDS_PATH)

                    for wsi_file in files:
                        wsi_path = os.path.join(self.save_dir, wsi_file)
                        wsi_id = os.path.splitext(wsi_file)[0]

                        if wsi_id not in self.fold_ids:
                            continue

                        if not wsi_path.endswith('.pt'):
                            continue

                        try:
                            wsi_features = torch.load(wsi_path)

                            if wsi_features.is_cuda:
                                wsi_features = wsi_features.cpu()

                            if wsi_features.dim() > 1:
                                final_features = wsi_features.flatten()
                            else:
                                final_features = wsi_features

                            # Get label from CSV
                            if task_name == "MSIH":
                                if wsi_id in folds_df['WSI_Id'].values:
                                    fold_label = folds_df.loc[folds_df['WSI_Id'] == wsi_id, 'label'].values[0]
                                    label = 0 if fold_label == 'nonMSIH' else 1
                                else:
                                    continue
                            else:
                                raise ValueError(f"Unknown task name: {task_name}")

                            self.data.append((final_features, label, wsi_id))

                        except Exception as e:
                            print(f"[ERROR] Loading failed for {wsi_path}: {e}")

                def __len__(self):
                    return len(self.data)

                def __getitem__(self, idx):
                    features, label, wsi_id = self.data[idx]
                    return features, label, wsi_id

            # ============================================
            # TRAIN AND EVALUATE FUNCTION
            # ============================================
            def train_and_evaluate(fold, train_loader, val_loader, test_loader, model_type='lin'):
                def get_feats_labels(loader):
                    feats, labels, ids = [], [], []
                    for features, lbls, wsi_ids in loader:
                        feats.append(features)
                        labels.append(lbls)
                        if isinstance(wsi_ids, (list, tuple)):
                            ids.extend(wsi_ids)
                        else:
                            ids.append(wsi_ids)
                    return torch.cat(feats), torch.cat(labels), ids

                train_feats, train_labels, _ = get_feats_labels(train_loader)
                val_feats, val_labels, _ = get_feats_labels(val_loader)
                test_feats, test_labels, all_test_ids = get_feats_labels(test_loader)

                best_metrics = None
                best_dump = None
                best_params = None

                if model_type == 'lin':
                    param_grid = {
                        'C': [10],
                        'max_iter': [1000]
                    }
                    for C, max_iter in product(param_grid['C'], param_grid['max_iter']):
                        eval_metrics, eval_dump = eval_linear(
                            fold=fold,
                            train_feats=train_feats,
                            train_labels=train_labels,
                            valid_feats=val_feats,
                            valid_labels=val_labels,
                            test_feats=test_feats,
                            test_labels=test_labels,
                            max_iter=max_iter,
                            save_path=MODEL_SAVE_PATH,
                            C=C,
                            verbose=False,
                        )
                        if not best_metrics or eval_metrics['lin_macro_f1'] > best_metrics['lin_macro_f1']:
                            best_metrics = eval_metrics
                            best_dump = eval_dump
                            best_params = (C, max_iter)

                elif model_type == 'ann':
                    param_grid = {
                        'hidden_dim1': [128, 256],
                        'hidden_dim2': [128],
                        'max_iter': [300, 500]
                    }
                    for h1, h2, max_iter in product(param_grid['hidden_dim1'], param_grid['hidden_dim2'], param_grid['max_iter']):
                        eval_metrics, eval_dump = eval_ANN(
                            fold=fold,
                            train_feats=train_feats,
                            train_labels=train_labels,
                            valid_feats=val_feats,
                            valid_labels=val_labels,
                            test_feats=test_feats,
                            test_labels=test_labels,
                            input_dim=VECTOR_DIM,
                            hidden_dim=h1,
                            hidden_dim2=h2,
                            max_iter=max_iter,
                            model_save_path=MODEL_SAVE_PATH,
                            verbose=False,
                        )
                        if not best_metrics or eval_metrics['ann_macro_f1'] > best_metrics['ann_macro_f1']:
                            best_metrics = eval_metrics
                            best_dump = eval_dump
                            best_params = (h1, h2, max_iter)

                elif model_type == 'knn':
                    for k in [3]:
                        eval_metrics, eval_dump = eval_knn(
                            fold=fold,
                            train_feats=train_feats,
                            train_labels=train_labels,
                            val_feats=val_feats,
                            val_labels=val_labels,
                            test_feats=test_feats,
                            test_labels=test_labels,
                            n_neighbors=k,
                            normalize_feats=True,
                            model_save_path=MODEL_SAVE_PATH,
                            verbose=False
                        )
                        if not best_metrics or eval_metrics['knn_macro_f1'] > best_metrics['knn_macro_f1']:
                            best_metrics = eval_metrics
                            best_dump = eval_dump
                            best_params = k

                elif model_type == 'proto':
                    eval_metrics, eval_dump = eval_protonet(
                        fold=fold,
                        train_feats=train_feats,
                        train_labels=train_labels,
                        val_feats=val_feats,
                        val_labels=val_labels,
                        test_feats=test_feats,
                        test_labels=test_labels,
                        normalize_feats=True,
                        model_save_path=MODEL_SAVE_PATH,
                    )
                    best_metrics = eval_metrics
                    best_dump = eval_dump
                    best_params = None

                elif model_type == 'rf':
                    param_grid = {
                        'n_estimators': [500],
                        'max_depth': [None],
                        'min_samples_split': [5],
                        'min_samples_leaf': [1],
                        'class_weight': [{0: 1, 1: 10}],
                        'prediction_threshold': [0.3]
                    }
                    for n_est, max_d, min_split, min_leaf, cw, thresh in product(
                        param_grid['n_estimators'],
                        param_grid['max_depth'],
                        param_grid['min_samples_split'],
                        param_grid['min_samples_leaf'],
                        param_grid['class_weight'],
                        param_grid['prediction_threshold']
                    ):
                        eval_metrics, eval_dump = eval_r_forest(
                            fold=fold,
                            train_feats=train_feats,
                            train_labels=train_labels,
                            valid_feats=val_feats,
                            valid_labels=val_labels,
                            test_feats=test_feats,
                            test_labels=test_labels,
                            n_estimators=n_est,
                            max_depth=max_d,
                            min_samples_split=min_split,
                            min_samples_leaf=min_leaf,
                            class_weight=cw,
                            prediction_threshold=thresh,
                            save_path=MODEL_SAVE_PATH,
                            verbose=False,
                        )
                        if not best_metrics or eval_metrics['rf_macro_f1'] > best_metrics['rf_macro_f1']:
                            best_metrics = eval_metrics
                            best_dump = eval_dump
                            best_params = (n_est, max_d, min_split, min_leaf, cw, thresh)

                else:
                    raise ValueError(f"Unsupported model type: {model_type}")

                if best_params:
                    print(f"Fold {fold} | Best Params for {model_type}: {best_params}")
                return best_metrics, best_dump, all_test_ids

            # ============================================
            # K-FOLD CROSS VALIDATION
            # ============================================
            def run_k_fold_cross_validation(save_dir: str, folds: List[List[str]], model_type: str = 'lin'):
                results_per_fold = []
                num_folds = len(folds)

                for i in range(num_folds):
                    test_ids = folds[i]
                    val_ids = folds[(i + 1) % num_folds]

                    train_ids = []
                    for j in range(num_folds):
                        if j != i and j != (i + 1) % num_folds:
                            train_ids.extend(folds[j])

                    print(f"Running Fold {i + 1} with model {model_type}...")

                    train_dataset = WSIDataset(save_dir, train_ids)
                    val_dataset = WSIDataset(save_dir, val_ids)
                    test_dataset = WSIDataset(save_dir, test_ids)

                    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
                    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
                    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

                    eval_metrics, eval_dump, all_test_ids = train_and_evaluate(i, train_loader, val_loader, test_loader, model_type=model_type)
                    print_metrics(eval_metrics)

                    result = {
                        **eval_metrics,
                        **eval_dump,
                        "wsi_ids": all_test_ids,
                        "fold": i + 1
                    }
                    results_per_fold.append(result)

                return results_per_fold

            # ============================================
            # LOAD FOLDS AND RUN PIPELINE
            # ============================================
            # 1. Load the CSV
            all_csv_data = pd.read_csv(K_FOLDS_PATH)
            csv_wsi_ids = all_csv_data['WSI_Id'].tolist()

            # 2. Filter only IDs that actually exist as .pt files in DATA_PATH
            existing_files = [os.path.splitext(f)[0] for f in os.listdir(DATA_PATH) if f.endswith('.pt')]
            valid_wsi_ids = [w_id for w_id in csv_wsi_ids if w_id in existing_files]

            print(f"Found {len(csv_wsi_ids)} entries in CSV.")
            print(f"Verified {len(valid_wsi_ids)} physical files exist in {DATA_PATH}.")

            # 3. Split into 4 folds manually
            num_total_slides = len(valid_wsi_ids)
            num_folds = 4
            folds = []

            # Calculate fold sizes
            fold_size = num_total_slides // num_folds
            for i in range(num_folds):
                start_idx = i * fold_size
                # If it's the last fold, take all remaining slides
                if i == num_folds - 1:
                    fold_ids = valid_wsi_ids[start_idx:]
                else:
                    fold_ids = valid_wsi_ids[start_idx: start_idx + fold_size]
                folds.append(fold_ids)
                print(f"Fold {i+1} created with {len(fold_ids)} slides.")

            # Run k-fold cross-validation with different models
            model_types = ['lin', 'ann', 'knn', 'proto', 'rf']
            metric_indices = {
                'acc': 0,
                'bacc': 1,
                'macro_f1': 2,
                'weighted_f1': 3,
                'auroc': 4
            }

            eval_metrics__for_excel = []
            probs_all_for_excel = None

            for model in model_types:
                print(f"\n\n ********* Training with model: {model}********* \n\n")
                k_folds_results = run_k_fold_cross_validation(DATA_PATH, folds, model_type=model)
                model_df = build_probs_df(k_folds_results, model_name=model)

                if probs_all_for_excel is None:
                    probs_all_for_excel = model_df
                else:
                    probs_all_for_excel = pd.merge(probs_all_for_excel, model_df, on=["Fold", "WSI_ID", "Target"], how="outer")

                average_results = calculate_metric_averages(
                    [{k: v for k, v in result.items() if k in [f"{model}_{m}" for m in metric_indices.keys()]}
                     for result in k_folds_results],
                    metric_indices,
                    model_prefix=model
                )

                confusion_matrices = [np.array(result[f"{model}_conf_matrix"]) for result in k_folds_results if f"{model}_conf_matrix" in result]
                avg_conf_matrix = average_confusion_matrices(confusion_matrices)

                print("\n\n Average results for all folds:")
                for metric, value in average_results.items():
                    print(f"{metric}: {value:.4f}")

                for metric in metric_indices.keys():
                    row = [f"{model}_{metric}"]
                    for result in k_folds_results:
                        row.append(result.get(f"{model}_{metric}", 'N/A'))
                    row.append(average_results.get(f"{model}_{metric}", 'N/A'))
                    eval_metrics__for_excel.append(row)

                row = [f"{model}_conf_matrix"]
                for result in k_folds_results:
                    row.append(str(result.get(f"{model}_conf_matrix", "N/A")))
                row.append(str(avg_conf_matrix))
                eval_metrics__for_excel.append(row)

            eval_metrics_df = pd.DataFrame(eval_metrics__for_excel,
                                            columns=["Metric", "Fold1", "Fold2", "Fold3", "Fold4", "AvgFolds"])
            write_data_in_excel(EVAL_METRICS_EXCEL, eval_metrics_df, sheet_name=sheet_name)
            write_data_in_excel(PROBS_ALL_EXCEL, probs_all_for_excel, sheet_name=sheet_name)

print("✅ PAIP Pipeline Complete!")


  Aggregation: Caption_based_aggregation_15_classes | Model: H-Optimus-1

Found 78 entries in CSV.
Verified 73 physical files exist in D:\Aamir Gulzar\KSA_project2\paip_data\slide_aggregation\Caption_based_aggregation_15_classes\H-Optimus-1.
Fold 1 created with 18 slides.
Fold 2 created with 18 slides.
Fold 3 created with 18 slides.
Fold 4 created with 19 slides.


 ********* Training with model: lin********* 


Running Fold 1 with model lin...
Fold 0 | Best Params for lin: (10, 1000)
lin_acc: 0.9444
lin_bacc: 0.9643
lin_macro_f1: 0.9259
lin_weighted_f1: 0.9465
lin_auroc: 1.0000
lin_conf_matrix: [[13  1]
 [ 0  4]]
Running Fold 2 with model lin...
Fold 1 | Best Params for lin: (10, 1000)
lin_acc: 0.8333
lin_bacc: 0.6250
lin_macro_f1: 0.6516
lin_weighted_f1: 0.7914
lin_auroc: 0.9286
lin_conf_matrix: [[14  0]
 [ 3  1]]
Running Fold 3 with model lin...
Fold 2 | Best Params for lin: (10, 1000)
lin_acc: 1.0000
lin_bacc: 1.0000
lin_macro_f1: 1.0000
lin_weighted_f1: 1.0000
lin_auroc: 1.0000
l

### TCGA test ann and linear hyperparameters

In [4]:
    #Test ann for pipeline 2
    import pandas as pd
    import itertools
    import os

    # Grid of hyperparameters
    HIDDEN_DIM1_VALUES = [128, 256, 512]
    HIDDEN_DIM2_VALUES = [64, 128, 256]
    MAX_ITER_VALUES = [100, 200, 350, 500, 1000]

    # Storage
    grid_results = []

    # Grid Search over (hidden_dim1, hidden_dim2, max_iter)
    for hidden_dim1, hidden_dim2, max_iter in itertools.product(HIDDEN_DIM1_VALUES, HIDDEN_DIM2_VALUES, MAX_ITER_VALUES):
        print(f"\n🧪 Running CV with HIDDEN_DIM1={hidden_dim1}, HIDDEN_DIM2={hidden_dim2}, MAX_ITER={max_iter}")

        try:
            # Override the ANN tuning loop using these globals
            def custom_train_and_evaluate(fold, train_loader, val_loader, test_loader, model_type='ann'):
                all_feats = lambda loader: zip(*[(x, y, z) for x, y, z in loader])
                train_feats, train_labels, _ = all_feats(train_loader)
                val_feats, val_labels, _ = all_feats(val_loader)
                test_feats, test_labels, all_test_ids = all_feats(test_loader)

                train_feats = torch.cat(train_feats)
                train_labels = torch.cat(train_labels)
                val_feats = torch.cat(val_feats)
                val_labels = torch.cat(val_labels)
                test_feats = torch.cat(test_feats)
                test_labels = torch.cat(test_labels)

                eval_metrics, eval_dump = eval_ANN(
                    fold=fold,
                    train_feats=train_feats,
                    train_labels=train_labels,
                    valid_feats=val_feats,
                    valid_labels=val_labels,
                    test_feats=test_feats,
                    test_labels=test_labels,
                    input_dim=VECTOR_DIM,
                    hidden_dim=hidden_dim1,
                    hidden_dim2=hidden_dim2,
                    max_iter=max_iter,
                    model_save_path=MODEL_SAVE_PATH,
                    verbose=False,
                )
                return eval_metrics, eval_dump, all_test_ids

            # Inject into main loop temporarily
            original_train_and_evaluate = globals()['train_and_evaluate']
            globals()['train_and_evaluate'] = custom_train_and_evaluate

            # Run CV
            results = run_k_fold_cross_validation(
                save_dir=DATA_PATH,
                folds=folds,
                model_type='ann'
            )

            # Restore original function
            globals()['train_and_evaluate'] = original_train_and_evaluate

            # Extract avg metrics
            avg_acc = sum([r.get("ann_bacc", 0) for r in results]) / len(results)
            avg_auc = sum([r.get("ann_auroc", 0) for r in results]) / len(results)

            # Save result
            grid_results.append({
                "HIDDEN_DIM1": hidden_dim1,
                "HIDDEN_DIM2": hidden_dim2,
                "MAX_ITER": max_iter,
                "Avg Accuracy": avg_acc,
                "Avg AUC": avg_auc
            })

        except Exception as e:
            print(f"❌ Error with HIDDEN_DIM1={hidden_dim1}, HIDDEN_DIM2={hidden_dim2}, MAX_ITER={max_iter}: {e}")
            grid_results.append({
                "HIDDEN_DIM1": hidden_dim1,
                "HIDDEN_DIM2": hidden_dim2,
                "MAX_ITER": max_iter,
                "Avg Accuracy": None,
                "Avg AUC": None,
                "Error": str(e)
            })
            globals()['train_and_evaluate'] = original_train_and_evaluate  # Restore in case of exception

    # Save to Excel
    df_grid = pd.DataFrame(grid_results)
    excel_path = os.path.join(OUTPUT_SAVE_PATH, "grid_search_ann_hidden1_hidden2_iter.xlsx")
    df_grid.to_excel(excel_path, index=False)

    # Display best config
    df_sorted = df_grid.sort_values(by="Avg Accuracy", ascending=False)
    best_row = df_sorted.iloc[0]
    print("\n✅ Grid search complete. Best configuration:")
    print(best_row)

    df_grid

    #Test linear for pipeline 2
    import pandas as pd
    import itertools
    import os

    # Grid of hyperparameters for Logistic Regression
    C_VALUES = [0.01, 0.1, 1, 10]
    MAX_ITER_VALUES = [300, 500, 1000]

    # Storage
    grid_results = []

    # Grid Search over (C, max_iter)
    for C, max_iter in itertools.product(C_VALUES, MAX_ITER_VALUES):
        print(f"\n🧪 Running CV with C={C}, MAX_ITER={max_iter}")
        
        try:
            # Override the linear tuning loop using these globals
            def custom_train_and_evaluate(fold, train_loader, val_loader, test_loader, model_type='lin'):
                all_feats = lambda loader: zip(*[(x, y, z) for x, y, z in loader])
                train_feats, train_labels, *_ = all_feats(train_loader)
                val_feats, val_labels, *_ = all_feats(val_loader)
                test_feats, test_labels, all_test_ids = all_feats(test_loader)
                
                train_feats = torch.cat(train_feats)
                train_labels = torch.cat(train_labels)
                val_feats = torch.cat(val_feats)
                val_labels = torch.cat(val_labels)
                test_feats = torch.cat(test_feats)
                test_labels = torch.cat(test_labels)
                
                eval_metrics, eval_dump = eval_linear(
                    fold=fold,
                    train_feats=train_feats,
                    train_labels=train_labels,
                    valid_feats=val_feats,
                    valid_labels=val_labels,
                    test_feats=test_feats,
                    test_labels=test_labels,
                    max_iter=max_iter,
                    save_path=MODEL_SAVE_PATH,
                    C=C,
                    verbose=False,
                )
                return eval_metrics, eval_dump, all_test_ids
            
            # Inject into main loop temporarily
            original_train_and_evaluate = globals()['train_and_evaluate']
            globals()['train_and_evaluate'] = custom_train_and_evaluate
            
            # Run CV
            results = run_k_fold_cross_validation(
                save_dir=DATA_PATH,
                folds=folds,
                model_type='lin'
            )
            
            # Restore original function
            globals()['train_and_evaluate'] = original_train_and_evaluate
            
            # Extract avg metrics (using bacc as key metric)
            avg_bacc = sum([r.get("lin_bacc", 0) for r in results]) / len(results)
            avg_auc = sum([r.get("lin_auroc", 0) for r in results]) / len(results)
            avg_f1 = sum([r.get("lin_macro_f1", 0) for r in results]) / len(results)
            
            # Save result
            grid_results.append({
                "C": C,
                "MAX_ITER": max_iter,
                "Avg BACC": avg_bacc,
                "Avg AUC": avg_auc,
                "Avg F1": avg_f1
            })
            
        except Exception as e:
            print(f"❌ Error with C={C}, MAX_ITER={max_iter}: {e}")
            grid_results.append({
                "C": C,
                "MAX_ITER": max_iter,
                "Avg BACC": None,
                "Avg AUC": None,
                "Avg F1": None,
                "Error": str(e)
            })
            globals()['train_and_evaluate'] = original_train_and_evaluate  # Restore in case of exception

    # Save to Excel
    df_grid = pd.DataFrame(grid_results)
    excel_path = os.path.join(OUTPUT_SAVE_PATH, "grid_search_linear_C_maxiter.xlsx")
    df_grid.to_excel(excel_path, index=False)

    # Display best configuration (sorted by BACC)
    df_sorted = df_grid.sort_values(by="Avg BACC", ascending=False)
    best_row = df_sorted.iloc[0]
    print("\n✅ Linear Grid search complete. Best configuration:")
    print(best_row)
    df_grid


🧪 Running CV with HIDDEN_DIM1=128, HIDDEN_DIM2=64, MAX_ITER=100
Running Fold 1 with model ann...
  Train IDs count: 215, Val IDs count: 98, Test IDs count: 103
  Dataset sizes - Train: 213, Val: 97, Test: 103
ann_acc: 0.7864
ann_bacc: 0.8473
ann_macro_f1: 0.7095
ann_weighted_f1: 0.8154
ann_auroc: 0.9023
ann_conf_matrix: [[67 21]
 [ 1 14]]
Running Fold 2 with model ann...
  Train IDs count: 206, Val IDs count: 112, Test IDs count: 98
  Dataset sizes - Train: 204, Val: 112, Test: 97
ann_acc: 0.8454
ann_bacc: 0.8132
ann_macro_f1: 0.7385
ann_weighted_f1: 0.8609
ann_auroc: 0.8727
ann_conf_matrix: [[72 12]
 [ 3 10]]
Running Fold 3 with model ann...
  Train IDs count: 201, Val IDs count: 103, Test IDs count: 112
  Dataset sizes - Train: 200, Val: 101, Test: 112
ann_acc: 0.8304
ann_bacc: 0.8380
ann_macro_f1: 0.7652
ann_weighted_f1: 0.8447
ann_auroc: 0.8810
ann_conf_matrix: [[76 16]
 [ 3 17]]
Running Fold 4 with model ann...
  Train IDs count: 210, Val IDs count: 103, Test IDs count: 103
  Dat

,C,MAX_ITER,Avg BACC,Avg AUC,Avg F1
0,0.01,300,0.575318,0.845281,0.590929
1,0.01,500,0.575318,0.845281,0.590929
2,0.01,1000,0.575318,0.845281,0.590929
3,0.10,300,0.680520,0.867655,0.713570
4,0.10,500,0.680520,0.867655,0.713570
5,0.10,1000,0.680520,0.867655,0.713570
6,1.00,300,0.720560,0.856302,0.744748
7,1.00,500,0.720560,0.856030,0.744748
8,1.00,1000,0.720560,0.856166,0.744748
9,10.00,300,0.697649,0.840916,0.713317


## Pipeline for PAIP data

In [1]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
import numpy as np
from typing import Dict, List, Tuple, Any
import time
import torch
import joblib
import os
from torch.nn.functional import normalize

# ============================================
# FIXED eval_linear - No more combine_trainval
# ============================================
def eval_linear(
    fold: int,
    train_feats: torch.Tensor,
    train_labels: torch.Tensor,
    valid_feats: torch.Tensor,
    valid_labels: torch.Tensor,
    test_feats: torch.Tensor,
    test_labels: torch.Tensor,
    max_iter: int = 1000,
    C: float = 1.0,
    prefix: str = "lin_",
    save_path: str = None,
    verbose: bool = True,
) -> Tuple[Dict[str, Any], Dict[str, Any]]:
    """
    Evaluate a linear logistic regression probe using Scikit-learn.
    
    FIXED: Removed combine_trainval parameter - now ONLY trains on train_feats
    """
    if verbose:
        print(f"Train Shape: {train_feats.shape}, Test Shape: {test_feats.shape}")
        if valid_feats is not None:
            print(f"Validation Shape: {valid_feats.shape}")
    
    start = time.time()
    
    # Train linear probe (ONLY on training data)
    classifier = train_linear_probe(
        train_feats,
        train_labels,
        max_iter=max_iter,
        C=C,
        verbose=verbose,
    )
    
    if save_path is not None:
        model_path = os.path.join(save_path, f"fold{fold}_logistic_regression.pkl")
        joblib.dump(classifier, model_path)
    
    # Test linear probe
    results, dump = test_linear_probe(classifier, test_feats, test_labels, prefix=prefix)
    
    if verbose:
        print(f"Linear Probe Evaluation Time: {time.time() - start:.3f} s")

    return results, dump


def train_linear_probe(
    train_feats: torch.Tensor,
    train_labels: torch.Tensor,
    max_iter: int = 1000,
    C: float = 1.0,
    verbose: bool = True,
) -> LogisticRegression:
    """
    Train a logistic regression classifier using Scikit-learn.
    
    FIXED: Removed combine_trainval logic - trains ONLY on provided train_feats
    """
    if verbose:
        print(f"Training on {train_feats.shape[0]} samples")
    
    classifier = LogisticRegression(
        C=C, 
        max_iter=max_iter, 
        solver="lbfgs", 
        verbose=0,  # Changed to 0 to reduce clutter
        random_state=42
    )
    classifier.fit(train_feats, train_labels)
    
    if verbose:
        print(f"Training complete. Coefficients shape: {classifier.coef_.shape}")
    
    return classifier


def test_linear_probe(
    classifier: LogisticRegression,
    test_feats: torch.Tensor,
    test_labels: torch.Tensor,
    num_classes: int = None,
    prefix: str = "lin_",
) -> Tuple[Dict[str, Any], Dict[str, Any]]:
    """
    Evaluate a trained logistic regression classifier on the test set.
    """
    NUM_C = len(set(test_labels.cpu().numpy())) if num_classes is None else num_classes
    
    # Predict and get probabilities
    if NUM_C == 2:
        probs_all = classifier.predict_proba(test_feats)
        roc_kwargs = {}
    else:
        probs_all = classifier.predict_proba(test_feats)
        roc_kwargs = {"multi_class": "ovo", "average": "macro"}

    preds_all = classifier.predict(test_feats)
    targets_all = test_labels.detach().cpu().numpy()
    
    eval_metrics = get_eval_metrics(
        targets_all, preds_all, probs_all, True, prefix, roc_kwargs
    )
    dump = {
        "preds_all": preds_all, 
        "probs_all": probs_all, 
        "targets_all": targets_all
    }

    return eval_metrics, dump


# ============================================
# FIXED eval_knn - No more combining train+val
# ============================================
def eval_knn(
    fold: int,
    train_feats: torch.Tensor,
    train_labels: torch.Tensor,
    val_feats: torch.Tensor,
    val_labels: torch.Tensor,
    test_feats: torch.Tensor,
    test_labels: torch.Tensor,
    n_neighbors: int = 3,
    normalize_feats: bool = True,
    prefix: str = "knn_",
    model_save_path: str = None,
    verbose: bool = False,
) -> Tuple[Dict[str, Any], Dict[str, Any]]:
    """
    FIXED: Now trains ONLY on train_feats, validates on val_feats, tests on test_feats
    """
    # Normalize features if requested
    if normalize_feats:
        train_feats = normalize(train_feats, dim=-1, p=2)
        val_feats = normalize(val_feats, dim=-1, p=2)
        test_feats = normalize(test_feats, dim=-1, p=2)
    
    # Convert tensors to numpy for sklearn
    train_feats_np = train_feats.numpy()
    val_feats_np = val_feats.numpy()
    test_feats_np = test_feats.numpy()
    train_labels_np = train_labels.numpy()
    val_labels_np = val_labels.numpy()
    test_labels_np = test_labels.numpy()
    
    if verbose:
        print(f"Training KNN on {len(train_labels_np)} samples")
        print(f"Validating on {len(val_labels_np)} samples")
    
    # Train KNN classifier with GridSearchCV on train, validate on val
    param_grid = {
        'n_neighbors': [3, 5, 7, 10, 15],
        'metric': ['cosine', 'euclidean', 'minkowski'],
        'weights': ['uniform', 'distance']
    }
    
    knn = GridSearchCV(
        KNeighborsClassifier(),
        param_grid,
        n_jobs=-1,
        verbose=0,
        scoring='balanced_accuracy',
        cv=3  # Use 3-fold CV within training set
    )
    
    knn.fit(train_feats_np, train_labels_np)
    
    if verbose:
        print(f"Best Params: {knn.best_params_}")
        print(f"Best Balanced Accuracy score: {knn.best_score_}")
    
    knn = knn.best_estimator_
    
    if model_save_path is not None:
        model_path = os.path.join(model_save_path, f"fold{fold}_knn_model.pkl")
        joblib.dump(knn, model_path)
    
    # Predict and evaluate on TEST set
    predicted_labels = knn.predict(test_feats_np)
    probs_all = knn.predict_proba(test_feats_np)
    
    metrics = get_eval_metrics(
        test_labels_np, predicted_labels, probs_all, prefix=prefix
    )
    
    dump = {
        "preds_all": predicted_labels,
        "probs_all": probs_all,
        "targets_all": test_labels_np
    }
    
    return metrics, dump


# ============================================
# Test saved model functions (unchanged)
# ============================================
def test_saved_logistic_model(
    test_feats: torch.Tensor, 
    test_labels: torch.Tensor, 
    model_path: str = "logistic_regression.pkl"
):
    """Load and test saved logistic regression model"""
    classifier = joblib.load(model_path)
    probs_all = classifier.predict_proba(test_feats)
    preds_all = classifier.predict(test_feats)
    targets_all = test_labels.cpu().numpy()
    
    eval_metrics = get_eval_metrics(
        targets_all, preds_all, probs_all, True, prefix="lin_"
    )
    dump = {
        "preds_all": preds_all, 
        "probs_all": probs_all, 
        "targets_all": targets_all
    }
    return eval_metrics, dump


def test_saved_knn_model(
    test_feats: torch.Tensor, 
    test_labels: torch.Tensor, 
    model_path: str = "knn_model.pkl"
):
    """Load and test saved KNN model"""
    knn = joblib.load(model_path)
    test_feats_np = test_feats.numpy()
    test_labels_np = test_labels.numpy()
    
    predicted_labels = knn.predict(test_feats_np)
    probs_all = knn.predict_proba(test_feats_np)
    
    eval_metrics = get_eval_metrics(
        test_labels_np, predicted_labels, probs_all, True, prefix="knn_"
    )
    dump = {
        "preds_all": predicted_labels, 
        "probs_all": probs_all, 
        "targets_all": test_labels_np
    }
    return eval_metrics, dump

In [14]:
import os
import torch
import pandas as pd
import numpy as np
import itertools
from torch.utils.data import Dataset, DataLoader
from typing import List
from openpyxl import load_workbook

# ============================================
# CONFIGS
# ============================================
VECTOR_DIM = 10752
HIDDEN_DIM = 128
BATCH_SIZE = 4
task_names = ["MSIH"]  # "MSIH", "BRAF", "KRAS", "TP53", "CIMP"
K_FOLDS_PATH = r"D:\Aamir Gulzar\KSA_project2\paip_data\paip_kfolds_71.csv"
DATA_PATH = r"D:\Aamir Gulzar\KSA_project2\paip_data\slide_aggregation\Caption_Based_Clustering_FiveCrop\conch_CC_fivecrop"
sheet_name = "baseline"

# ============================================
# DATASET CLASS
# ============================================
class WSIDataset(Dataset):
    def __init__(self, save_dir: str, fold_ids: List[str], task_name: str, folds_df: pd.DataFrame):
        self.data = []
        self.save_dir = save_dir
        self.fold_ids = fold_ids
        self.task_name = task_name
        self.folds_df = folds_df
        self._load_data()

    def _load_data(self):
        files = os.listdir(self.save_dir)

        for wsi_file in files:
            wsi_path = os.path.join(self.save_dir, wsi_file)
            wsi_id = os.path.splitext(wsi_file)[0]

            # Skip files not in fold_ids or not .pt files
            if wsi_id not in self.fold_ids or not wsi_path.endswith('.pt'):
                continue

            try:
                # Load WSI features
                wsi_features = torch.load(wsi_path)
                if wsi_features.is_cuda:
                    wsi_features = wsi_features.cpu()

                # Flatten features
                final_features = wsi_features.flatten() if wsi_features.dim() > 1 else wsi_features

                # Get label based on task
                if wsi_id not in self.folds_df['WSI_Id'].values:
                    continue

                if self.task_name == "MSIH":
                    fold_label = self.folds_df.loc[self.folds_df['WSI_Id'] == wsi_id, 'label'].values[0]
                    label = 0 if fold_label == 'nonMSIH' else 1
                elif self.task_name == "BRAF":
                    fold_label = self.folds_df.loc[self.folds_df['WSI_Id'] == wsi_id, 'BRAF_mutation'].values[0]
                    label = 0 if fold_label == 'WT' else 1
                elif self.task_name == "KRAS":
                    fold_label = self.folds_df.loc[self.folds_df['WSI_Id'] == wsi_id, 'KRAS_mutation'].values[0]
                    label = 0 if fold_label == 'WT' else 1
                elif self.task_name == "TP53":
                    fold_label = self.folds_df.loc[self.folds_df['WSI_Id'] == wsi_id, 'TP53_mutation'].values[0]
                    label = 0 if fold_label == 'WT' else 1
                else:
                    raise ValueError(f"Unknown task name: {self.task_name}")

                self.data.append((final_features, label, wsi_id))

            except Exception as e:
                print(f"[ERROR] Loading failed for {wsi_path}: {e}")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        features, label, wsi_id = self.data[idx]
        return features, label, wsi_id


# ============================================
# HELPER FUNCTIONS
# ============================================
def get_feats_labels(loader):
    """Extract features, labels, and IDs from dataloader"""
    feats, labels, ids = [], [], []
    for features, lbls, wsi_ids in loader:
        feats.append(features)
        labels.append(lbls)
        if isinstance(wsi_ids, (list, tuple)):
            ids.extend(wsi_ids)
        else:
            ids.append(wsi_ids)
    return torch.cat(feats), torch.cat(labels), ids


def write_data_in_excel(excel_path, df, sheet_name):
    """Write dataframe to Excel"""
    if os.path.exists(excel_path):
        with pd.ExcelWriter(excel_path, mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
            df.to_excel(writer, sheet_name=sheet_name, index=False)
    else:
        df.to_excel(excel_path, sheet_name=sheet_name, index=False)


# ============================================
# MAIN TRAINING LOOP
# ============================================
for task_name in task_names:
    print(f"\n{'='*60}\n🎯 Processing Task: {task_name}\n{'='*60}")
    
    # Setup paths
    MODEL_SAVE_PATH = os.path.join(
        r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\Slide_Classification\PAIP_Result\Caption_based_aggregation\Conch1_5",
        task_name, "models"
    )
    OUTPUT_SAVE_PATH = os.path.join(
        r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\Slide_Classification\PAIP_Result\Caption_based_aggregation\Conch1_5",
        task_name, "Output"
    )
    os.makedirs(MODEL_SAVE_PATH, exist_ok=True)
    os.makedirs(OUTPUT_SAVE_PATH, exist_ok=True)

    EVAL_METRICS_EXCEL = os.path.join(OUTPUT_SAVE_PATH, "cross_valid_avg_eval_metrics.xlsx")
    PROBS_ALL_EXCEL = os.path.join(OUTPUT_SAVE_PATH, "cross_valid_avg_probs_all.xlsx")

    # Load fold data
    if task_name == "MSIH":
        fold_path = K_FOLDS_PATH
    elif task_name == "BRAF":
        fold_path = r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\Slide_Classification\kfold_BRAF.csv"
    elif task_name == "KRAS":
        fold_path = r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\Slide_Classification\kfold_KRAS.csv"
    elif task_name == "TP53":
        fold_path = r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\Slide_Classification\kfold_TP53.csv"
    else:
        raise ValueError(f"Unknown task name: {task_name}")

    labels_df = pd.read_excel(fold_path) if fold_path.endswith(".xlsx") else pd.read_csv(fold_path)
    
    # Split into train and validation
    train_ids = labels_df[labels_df["WSI_Id"].str.contains("training", case=False)]["WSI_Id"].tolist()
    val_ids = labels_df[~labels_df["WSI_Id"].str.contains("training", case=False)]["WSI_Id"].tolist()
    
    print(f"✅ Found {len(train_ids)} training slides and {len(val_ids)} validation slides.")

    # Create datasets and loaders
    train_dataset = WSIDataset(DATA_PATH, train_ids, task_name, labels_df)
    val_dataset = WSIDataset(DATA_PATH, val_ids, task_name, labels_df)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

    # Extract features once
    train_feats, train_labels, _ = get_feats_labels(train_loader)
    val_feats, val_labels, val_ids_list = get_feats_labels(val_loader)

    print(f"📦 Data prepared: {len(train_dataset)} train, {len(val_dataset)} val samples.")

    # ============================================
    # SINGLE MODEL EVALUATION (No Grid Search)
    # ============================================
    print(f"\n{'='*60}\n🧪 Running Single Model Evaluation\n{'='*60}")
    
    model_types = ['lin', 'ann', 'knn', 'proto', 'rf']
    eval_metrics_for_excel = []
    probs_all_for_excel = None

    for model in model_types:
        print(f"\n🔬 Training {model.upper()} model...")
        
        if model == 'lin':
            eval_metrics, eval_dump = eval_linear(
                fold=1,
                train_feats=train_feats,
                train_labels=train_labels,
                valid_feats=val_feats,
                valid_labels=val_labels,
                test_feats=val_feats,  # Using validation as test for final evaluation
                test_labels=val_labels,
                max_iter=1000,
                save_path=MODEL_SAVE_PATH,
                C=10,
                verbose=True,
            )
        elif model == 'ann':
            eval_metrics, eval_dump = eval_ANN(
                fold=1,
                train_feats=train_feats,
                train_labels=train_labels,
                valid_feats=val_feats,
                valid_labels=val_labels,
                test_feats=val_feats,
                test_labels=val_labels,
                input_dim=VECTOR_DIM,
                hidden_dim=256,
                hidden_dim2=64,
                max_iter=500,
                model_save_path=MODEL_SAVE_PATH,
                verbose=True,
            )
        elif model == 'knn':
            eval_metrics, eval_dump = eval_knn(
                fold=1,
                train_feats=train_feats,
                train_labels=train_labels,
                val_feats=val_feats,
                val_labels=val_labels,
                test_feats=val_feats,
                test_labels=val_labels,
                n_neighbors=1,
                normalize_feats=True,
                model_save_path=MODEL_SAVE_PATH,
                verbose=True
            )
        elif model == 'proto':
            eval_metrics, eval_dump = eval_protonet(
                fold=1,
                train_feats=train_feats,
                train_labels=train_labels,
                val_feats=val_feats,
                val_labels=val_labels,
                test_feats=val_feats,
                test_labels=val_labels,
                normalize_feats=True,
                model_save_path=MODEL_SAVE_PATH,
            )
        elif model == 'rf':
            eval_metrics, eval_dump = eval_r_forest(
                fold=1,
                train_feats=train_feats,
                train_labels=train_labels,
                valid_feats=val_feats,
                valid_labels=val_labels,
                test_feats=val_feats,
                test_labels=val_labels,
                n_estimators=500,
                max_depth=None,
                min_samples_split=5,
                min_samples_leaf=1,
                class_weight={0: 1, 1: 10},
                prediction_threshold=0.3,
                save_path=MODEL_SAVE_PATH,
                verbose=True,
            )
        
        # Print metrics
        print_metrics(eval_metrics)
        
        # Store for Excel
        metric_names = ['acc', 'bacc', 'macro_f1', 'weighted_f1', 'auroc']
        for metric in metric_names:
            value = eval_metrics.get(f"{model}_{metric}", 'N/A')
            eval_metrics_for_excel.append([f"{model}_{metric}", value])
        
        # Add confusion matrix
        conf_matrix = eval_metrics.get(f"{model}_conf_matrix", "N/A")
        eval_metrics_for_excel.append([f"{model}_conf_matrix", str(conf_matrix)])

    # Save single model results
    eval_metrics_df = pd.DataFrame(eval_metrics_for_excel, columns=["Metric", "Value"])
    write_data_in_excel(EVAL_METRICS_EXCEL, eval_metrics_df, sheet_name=sheet_name)

    # ============================================
    # GRID SEARCH - ANN
    # ============================================
    print(f"\n{'='*60}\n🔍 ANN Grid Search\n{'='*60}")
    
    HIDDEN_DIM1_VALUES = [128, 256, 512]
    HIDDEN_DIM2_VALUES = [64, 128, 256]
    MAX_ITER_VALUES = [100, 200, 350, 500, 1000]
    
    ann_grid_results = []
    
    for hidden_dim1, hidden_dim2, max_iter in itertools.product(HIDDEN_DIM1_VALUES, HIDDEN_DIM2_VALUES, MAX_ITER_VALUES):
        print(f"Testing: h1={hidden_dim1}, h2={hidden_dim2}, iter={max_iter}")
        
        try:
            eval_metrics, _ = eval_ANN(
                fold=1,
                train_feats=train_feats,
                train_labels=train_labels,
                valid_feats=val_feats,
                valid_labels=val_labels,
                test_feats=val_feats,
                test_labels=val_labels,
                input_dim=VECTOR_DIM,
                hidden_dim=hidden_dim1,
                hidden_dim2=hidden_dim2,
                max_iter=max_iter,
                model_save_path=MODEL_SAVE_PATH,
                verbose=False,
            )
            
            # FIXED: Use balanced accuracy instead of accuracy
            avg_bacc = eval_metrics.get("ann_bacc", None)
            avg_auc = eval_metrics.get("ann_auroc", None)
            avg_f1 = eval_metrics.get("ann_macro_f1", None)
            
            ann_grid_results.append({
                "HIDDEN_DIM1": hidden_dim1,
                "HIDDEN_DIM2": hidden_dim2,
                "MAX_ITER": max_iter,
                "Balanced_Accuracy": avg_bacc,
                "AUC": avg_auc,
                "Macro_F1": avg_f1,
            })
        except Exception as e:
            print(f"❌ Error: {e}")
            ann_grid_results.append({
                "HIDDEN_DIM1": hidden_dim1,
                "HIDDEN_DIM2": hidden_dim2,
                "MAX_ITER": max_iter,
                "Balanced_Accuracy": None,
                "AUC": None,
                "Macro_F1": None,
                "Error": str(e),
            })
    
    df_ann_grid = pd.DataFrame(ann_grid_results)
    ann_excel_path = os.path.join(OUTPUT_SAVE_PATH, "grid_search_ann.xlsx")
    df_ann_grid.to_excel(ann_excel_path, index=False)
    
    # Display best config
    df_sorted = df_ann_grid.sort_values(by="Balanced_Accuracy", ascending=False)
    best_row = df_sorted.iloc[0]
    print(f"\n✅ Best ANN Config:\n{best_row}")

    # ============================================
    # GRID SEARCH - LINEAR
    # ============================================
    print(f"\n{'='*60}\n🔍 Linear Grid Search\n{'='*60}")
    
    C_VALUES = [0.01, 0.1, 1, 10]
    MAX_ITER_VALUES = [300, 500, 1000]
    
    lin_grid_results = []
    
    for C, max_iter in itertools.product(C_VALUES, MAX_ITER_VALUES):
        print(f"Testing: C={C}, iter={max_iter}")
        
        try:
            eval_metrics, _ = eval_linear(
                fold=1,
                train_feats=train_feats,
                train_labels=train_labels,
                valid_feats=val_feats,
                valid_labels=val_labels,
                test_feats=val_feats,
                test_labels=val_labels,
                max_iter=max_iter,
                save_path=MODEL_SAVE_PATH,
                C=C,
                verbose=False,
            )
            
            avg_bacc = eval_metrics.get("lin_bacc", None)
            avg_auc = eval_metrics.get("lin_auroc", None)
            avg_f1 = eval_metrics.get("lin_macro_f1", None)
            
            lin_grid_results.append({
                "C": C,
                "MAX_ITER": max_iter,
                "Balanced_Accuracy": avg_bacc,
                "AUC": avg_auc,
                "Macro_F1": avg_f1,
            })
        except Exception as e:
            print(f"❌ Error: {e}")
            lin_grid_results.append({
                "C": C,
                "MAX_ITER": max_iter,
                "Balanced_Accuracy": None,
                "AUC": None,
                "Macro_F1": None,
                "Error": str(e),
            })
    
    df_lin_grid = pd.DataFrame(lin_grid_results)
    lin_excel_path = os.path.join(OUTPUT_SAVE_PATH, "grid_search_linear.xlsx")
    df_lin_grid.to_excel(lin_excel_path, index=False)
    
    # Display best config
    df_sorted = df_lin_grid.sort_values(by="Balanced_Accuracy", ascending=False)
    best_row = df_sorted.iloc[0]
    print(f"\n✅ Best Linear Config:\n{best_row}")

print("\n🎉 All tasks completed!")


🎯 Processing Task: MSIH


FileNotFoundError: [Errno 2] No such file or directory: 'D:\\Aamir Gulzar\\KSA_project2\\paip_data\\paip_kfolds_71.csv'

## Pipeline for TCGA MultiClass classification (full)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import os
import pandas as pd
from typing import Optional, Tuple, List, Dict, Any, Union
from itertools import product
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import (
    balanced_accuracy_score,
    roc_auc_score,
    accuracy_score,
    classification_report,
    f1_score,
    confusion_matrix,
)

from openpyxl import load_workbook
from openpyxl.utils.dataframe import dataframe_to_rows
import os
# configs
VECTOR_DIM = 6912  # size of input feature vector, H-optimus-1 = 13824, UNI-2h = 13824, Conchv1_5 = 6912, TITAN = 768, Average_Features = 768
HIDDEN_DIM = 128   # size of ANN hidden layer
BATCH_SIZE = 4
task_name = "TCGA"  # "MSIH", "BRAF", "KRAS", "TP53", "CIMP"
K_FOLDS_PATH = r"D:\Aamir Gulzar\KSA_project2\paip_data\Results\paip_kfolds_71.csv"
DATA_PATH = r"D:\Aamir Gulzar\KSA_project2\paip_data\slide_aggregation\conch_average_fivecrop"
MODEL_SAVE_PATH = r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\Slide_Classification\PAIP_Result\Conch1_5" + "\\" + task_name + r"\models"
os.makedirs(MODEL_SAVE_PATH, exist_ok=True)
OUTPUT_SAVE_PATH = r"D:\Aamir Gulzar\KSA_project2\Cancer-detection-classifier\Slide_Classification\PAIP_Result\Conch1_5" + "\\" + task_name + r"\Output"
os.makedirs(OUTPUT_SAVE_PATH, exist_ok=True)
# create a excel sheet in the output folder to save the results
EVAL_METRICS_EXCEL = os.path.join(OUTPUT_SAVE_PATH, "cross_valid_avg_eval_metrics.xlsx")
PROBS_ALL_EXCEL = os.path.join(OUTPUT_SAVE_PATH, "cross_valid_avg_probs_all.xlsx")
sheet_name = "baseline"

class ANNMultiClassifier:
    def __init__(
        self,
        input_dim: int = 512,
        hidden_dim1: int = 512,
        hidden_dim2: int = 128,
        num_classes: int = 3,
        max_iter: int = 100,
        verbose: bool = True
    ):
        self.max_iter = max_iter
        self.verbose = verbose
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.input_dim = input_dim
        self.hidden_dim1 = hidden_dim1
        self.hidden_dim2 = hidden_dim2
        self.num_classes = num_classes

        # Model definition (NO softmax here)
        self.model = nn.Sequential(
            nn.Linear(self.input_dim, self.hidden_dim1),
            nn.ReLU(),
            nn.BatchNorm1d(self.hidden_dim1),
            nn.Dropout(0.3),
            nn.Linear(self.hidden_dim1, self.hidden_dim2),
            nn.ReLU(),
            nn.BatchNorm1d(self.hidden_dim2),
            nn.Dropout(0.3),
            nn.Linear(self.hidden_dim2, self.num_classes)
        ).to(self.device)

        self.loss_func = nn.CrossEntropyLoss()

    def compute_loss(self, preds: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        return self.loss_func(preds, labels)

    def predict_proba(self, feats: torch.Tensor) -> torch.Tensor:
        feats = feats.to(self.device)
        self.model.eval()
        with torch.no_grad():
            logits = self.model(feats)
            return torch.softmax(logits, dim=1)  # Apply softmax here for probabilities

    def fit(
        self,
        train_feats: torch.Tensor,
        train_labels: torch.Tensor,
        val_feats: Optional[torch.Tensor] = None,
        val_labels: Optional[torch.Tensor] = None,
        combine_trainval: bool = False
    ) -> Tuple[List[float], List[float]]:
        train_feats, train_labels = train_feats.to(self.device), train_labels.to(self.device)

        if val_feats is not None:
            val_feats, val_labels = val_feats.to(self.device), val_labels.to(self.device)

        if combine_trainval and val_feats is not None:
            train_feats = torch.cat([train_feats, val_feats], dim=0)
            train_labels = torch.cat([train_labels, val_labels], dim=0)
            val_feats, val_labels = None, None

        optimizer = optim.Adam(self.model.parameters(), lr=1e-4, weight_decay=1e-4)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.3, patience=5, verbose=self.verbose
        )

        train_loss_history, val_loss_history = [], []
        best_val_loss = float("inf")
        patience, epochs_no_improve = 20, 0

        for epoch in range(self.max_iter):
            # Training
            self.model.train()
            preds = self.model(train_feats)
            loss = self.compute_loss(preds, train_labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss_history.append(loss.item())

            # Validation
            val_loss = None
            if val_feats is not None and not combine_trainval:
                self.model.eval()
                with torch.no_grad():
                    val_preds = self.model(val_feats)
                    val_loss = self.compute_loss(val_preds, val_labels)
                val_loss_history.append(val_loss.item())

                # Early stopping
                if val_loss.item() < best_val_loss:
                    best_val_loss = val_loss.item()
                    epochs_no_improve = 0
                else:
                    epochs_no_improve += 1
                    if epochs_no_improve >= patience:
                        if self.verbose:
                            print(f"Early stopping at epoch {epoch}")
                        break

            # Scheduler step
            scheduler.step(val_loss.item() if val_loss is not None else loss.item())

            if self.verbose and epoch % 10 == 0:
                if val_loss is not None:
                    print(f"Epoch {epoch}: Train Loss = {loss:.4f}, Val Loss = {val_loss:.4f}")
                else:
                    print(f"Epoch {epoch}: Train Loss = {loss:.4f}")

        return train_loss_history, val_loss_history


def get_eval_metrics(
    targets_all: Union[List[int], np.ndarray],
    preds_all: Union[List[int], np.ndarray],
    probs_all: Optional[Union[List[float], np.ndarray]] = None,
    get_report: bool = True,
    prefix: str = "",
    roc_kwargs: Dict[str, Any] = {},
) -> Dict[str, Any]:
    """
    Calculate evaluation metrics for binary or multi-class classification.
    """
    bacc = balanced_accuracy_score(targets_all, preds_all)
    acc = accuracy_score(targets_all, preds_all)
    macro_f1 = f1_score(targets_all, preds_all, average="macro")
    cls_rep = classification_report(targets_all, preds_all, output_dict=True, zero_division=0)
    conf_matrix = confusion_matrix(targets_all, preds_all)
    
    eval_metrics = {
        f"{prefix}acc": acc,
        f"{prefix}bacc": bacc,
        f"{prefix}macro_f1": macro_f1,
        f"{prefix}weighted_f1": cls_rep["weighted avg"]["f1-score"],
    }
    
    if probs_all is not None:
        try:
            n_classes = probs_all.shape[1]
            if n_classes > 2:
                # Multi-class ROC AUC (One-vs-Rest)
                roc_auc = roc_auc_score(
                    targets_all,
                    probs_all,
                    multi_class="ovr",
                    average="macro",
                    **roc_kwargs
                )
            else:
                # Binary ROC AUC
                roc_auc = roc_auc_score(targets_all, probs_all[:, 1], **roc_kwargs)
            eval_metrics[f"{prefix}auroc"] = roc_auc
        except ValueError:
            eval_metrics[f"{prefix}auroc"] = np.nan
    
    eval_metrics[f"{prefix}conf_matrix"] = conf_matrix
    return eval_metrics


def print_metrics(eval_metrics):
    for key, value in eval_metrics.items():
        if isinstance(value, (int, float)):
            print(f"{key}: {value:.4f}")
        else:
            print(f"{key}: {value}")


def eval_ANN(
    fold: int,
    train_feats: torch.Tensor,
    train_labels: torch.Tensor,
    valid_feats: torch.Tensor,
    valid_labels: torch.Tensor,
    test_feats: torch.Tensor,
    test_labels: torch.Tensor,
    input_dim: int = 512,
    hidden_dim: int = 512,
    hidden_dim2: int = 128,
    max_iter: int = 1000,
    num_classes: int = 3,
    prefix: str = "ann_",
    combine_trainval: bool = False,
    model_save_path: str = None,
    verbose: bool = False,
) -> Tuple[dict, dict]:

    if verbose:
        print(f"[Fold {fold}] Train Shape: {train_feats.shape}, Test Shape: {test_feats.shape}")

    classifier = ANNMultiClassifier(
        input_dim=input_dim,
        hidden_dim1=hidden_dim,
        hidden_dim2=hidden_dim2,
        num_classes=num_classes,
        max_iter=max_iter,
        verbose=verbose
    )

    classifier.fit(train_feats, train_labels, valid_feats, valid_labels, combine_trainval)

    if model_save_path is not None:
        os.makedirs(model_save_path, exist_ok=True)
        model_path = os.path.join(model_save_path, f"fold{fold}_trained_ann_model_{input_dim}.pth")
        torch.save(classifier.model.state_dict(), model_path)

    probs_all = classifier.predict_proba(test_feats).cpu().numpy()
    preds_all = np.argmax(probs_all, axis=1)
    targets_all = test_labels.cpu().numpy()

    metrics = get_eval_metrics(
        targets_all=targets_all,
        preds_all=preds_all,
        probs_all=probs_all,
        prefix=prefix
    )

    dump = {
        "preds_all": preds_all,
        "probs_all": probs_all,
        "targets_all": targets_all
    }

    return metrics, dump


def get_feats_labels(loader):
    feats, labels, ids = [], [], []
    for features, lbls, wsi_ids in loader:
        feats.append(features)
        labels.append(lbls)
        if isinstance(wsi_ids, (list, tuple)):
            ids.extend(wsi_ids)
        else:
            ids.append(wsi_ids)
    return torch.cat(feats), torch.cat(labels), ids


# Global variables for tracking averages
avg_fold_metrics = {}
classifier_fold_count = {}


def save_results_to_csv(
    save_path: str,
    classifier: str,
    fold: Union[int, str],
    eval_metrics: Dict[str, Any],
    num_folds: int = 4,
):
    global avg_fold_metrics, classifier_fold_count
    
    os.makedirs(save_path, exist_ok=True)
    results_file_path = os.path.join(save_path, "results.csv")
    
    # Check if file exists, if not create with header
    if not os.path.exists(results_file_path):
        with open(results_file_path, "w") as f:
            f.write("classifier,fold,Accuracy,Balanced Accuracy,Macro-F1,Weighted F1-Score,AUROC\n")
    
    # Append results
    with open(results_file_path, "a") as f:
        f.write(
            f"{classifier},{fold},{eval_metrics['ann_acc']:.4f},{eval_metrics['ann_bacc']:.4f},"
            f"{eval_metrics['ann_macro_f1']:.4f},{eval_metrics['ann_weighted_f1']:.4f},"
            f"{eval_metrics['ann_auroc']:.4f}\n"
        )
        
        # Track metrics for averaging
        if classifier not in avg_fold_metrics:
            avg_fold_metrics[classifier] = {k: 0 for k in eval_metrics.keys()}
            classifier_fold_count[classifier] = 0
            
        for k, v in eval_metrics.items():
            if isinstance(v, (int, float)):
                avg_fold_metrics[classifier][k] += v
        classifier_fold_count[classifier] += 1
        
        # Write average if we've completed all folds
        if classifier_fold_count[classifier] == num_folds:
            avg_results = {k: v / num_folds for k, v in avg_fold_metrics[classifier].items() if isinstance(v, (int, float))}
            f.write(
                f"{classifier},Average,{avg_results['ann_acc']:.4f},{avg_results['ann_bacc']:.4f},"
                f"{avg_results['ann_macro_f1']:.4f},{avg_results['ann_weighted_f1']:.4f},"
                f"{avg_results['ann_auroc']:.4f}\n"
            )
            # Reset tracking
            avg_fold_metrics.pop(classifier)
            classifier_fold_count.pop(classifier)


def main():
    # Load CSV containing fold and wsi_id
    folds_df = pd.read_csv(K_FOLDS_PATH)

    # Ensure correct column names
    assert 'fold' in folds_df.columns and 'WSI_Id' in folds_df.columns, "CSV must have 'fold' and 'WSI_Id' columns"

    # Group wsi_ids by fold
    num_folds = folds_df['fold'].nunique()
    folds = []
    for fold_num in sorted(folds_df['fold'].unique()):
        wsi_ids = folds_df[folds_df['fold'] == fold_num]['WSI_Id'].tolist()
        folds.append(wsi_ids)

    results_per_fold = []
    save_dir = DATA_PATH

    # Grid Search parameters for ANN
    param_grid = {
        'hidden_dim1': [128, 256, 512],
        'hidden_dim2': [64, 128, 256],
        'max_iter': [300, 500, 1000]
    }

    for i in range(num_folds):
        # Define test and validation folds
        test_ids = folds[i]
        val_ids = folds[(i + 1) % num_folds]

        # Use remaining folds as training
        train_ids = []
        for j in range(num_folds):
            if j != i and j != (i + 1) % num_folds:
                train_ids.extend(folds[j])
                
        print(f"Running Fold {i + 1} with model ann...")
        
        # Create datasets and loaders
        train_dataset = WSIDataset(save_dir, train_ids)
        val_dataset = WSIDataset(save_dir, val_ids)
        test_dataset = WSIDataset(save_dir, test_ids)
        
        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
        test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
        
        train_feats, train_labels, _ = get_feats_labels(train_loader)
        val_feats, val_labels, _ = get_feats_labels(val_loader)
        test_feats, test_labels, all_test_ids = get_feats_labels(test_loader)

        # Grid Search for ANN
        best_metrics = None
        best_dump = None
        best_params = None

        for h1, h2, max_iter in product(param_grid['hidden_dim1'], param_grid['hidden_dim2'], param_grid['max_iter']):
            eval_metrics, eval_dump = eval_ANN(
                fold=i,
                train_feats=train_feats,
                train_labels=train_labels,
                valid_feats=val_feats,
                valid_labels=val_labels,
                test_feats=test_feats,
                test_labels=test_labels,
                input_dim=VECTOR_DIM,
                hidden_dim=h1,
                hidden_dim2=h2,
                max_iter=max_iter,
                model_save_path=MODEL_SAVE_PATH,
                verbose=True,
            )

            # Use balanced_accuracy as the primary selection metric
            if not best_metrics or eval_metrics['ann_bacc'] > best_metrics['ann_bacc']:
                best_metrics = eval_metrics
                best_dump = eval_dump
                best_params = (h1, h2, max_iter)

        print(f"Best ANN params for fold {i}: "
              f"h1={best_params[0]}, h2={best_params[1]}, max_iter={best_params[2]}")
        
        # Save results
        save_results_to_csv(
            save_path=MODEL_SAVE_PATH,
            classifier="ann",
            fold=i+1,
            eval_metrics=best_metrics,
            num_folds=num_folds
        )
        
        results_per_fold.append({
            'fold': i+1,
            'metrics': best_metrics,
            'dump': best_dump,
            'best_params': best_params
        })

    print("Training completed for all folds!")
    return results_per_fold


if __name__ == "__main__":
    main()

## Analyze pt file

In [ ]:
import torch

# Path to your file
file_path = r"D:\Aamir Gulzar\KSA_project2\paip_data\slide_aggregation\Averaging\uni2_average_fivecrop\training_data_01_MSIH.pt"

# Load the file
data = torch.load(file_path, map_location='cpu')

# Check the type of the loaded object
print(f"Type of loaded object: {type(data)}")

# If it’s a dictionary (common for datasets or checkpoints), list its keys
if isinstance(data, dict):
    print("\nKeys in the loaded dictionary:")
    for k in data.keys():
        print(f" - {k}")

    # Optionally print shapes or summary info for each key
    print("\nSummary of each key:")
    for k, v in data.items():
        if torch.is_tensor(v):
            print(f"{k}: tensor of shape {tuple(v.shape)} and dtype {v.dtype}")
        elif isinstance(v, dict):
            print(f"{k}: nested dict with {len(v)} keys")
        elif isinstance(v, list):
            print(f"{k}: list with {len(v)} elements")
        else:
            print(f"{k}: type {type(v)}")

# If it’s a tensor
elif torch.is_tensor(data):
    print(f"\nIt’s a tensor with shape {data.shape} and dtype {data.dtype}")

# If it’s a model
elif hasattr(data, 'state_dict'):
    print("\nIt looks like a model. Here's its state_dict keys:")
    for k in data.state_dict().keys():
        print(f" - {k}")

# For any other type
else:
    print("\nUnrecognized type. Here’s the direct object:")
    print(data)

